# Quinta Playa GCP Offset Correction

Reconciles DEMs and point clouds georeferenced against different GCP coordinate files onto one common frame.

**The problem:** Two GCP coordinate files disagree by a systematic offset:
- **set 1** = `ground_control_points.txt`
- **set 2** = `GCP_2025.txt` / `SHAPE_CONTROL/CONTROL_HITOS_TACHOS_DISCOS.txt`

(A third source, **set 3**, is hand-written field coordinates — it's noisy by nature, not a rigid-shift candidate, and is not used for correction here.)

Each month's DEM/point cloud was georeferenced against whichever set fit that month's flight best, so different months currently sit on two different absolute reference frames.

**Master frame: set 2** (`GCP_2025.txt`). Files georeferenced against set 1 get converted onto set 2's coordinates (`direction="set1_to_set2"`, which is what's pre-filled in the "Apply it" cells below) — files already on set 2 need no correction.

This notebook:

1. Reports the verified **SET_1 ↔ SET_2 offset** (dx east, dy north, dz up) and per-point residuals, taken directly from the verified comparison spreadsheet (`gcp_comparison_results.xlsx`, Sept 2026).
2. Applies that offset to a **DEM GeoTIFF** — horizontal shift by rewriting the raster's affine transform (exact, no resampling); vertical shift by adding a constant to every valid pixel.
3. Applies that offset to a **point cloud (.las/.laz)** — adds dx/dy/dz directly to every point's X/Y/Z. No resampling either way; classification, intensity, RGB, etc. pass through unchanged.

**Important caveats — read before trusting the output:**

- The SET_1 ↔ SET_2 offset below is taken directly from the user's own verified comparison (`gcp_comparison_results.xlsx`), not re-derived here. Residual scatter across the 10 matched points is ~3–4 cm on dx/dz and ~4 cm on dy — tight enough that this looks like a genuine uniform rigid shift, not a rotation/scale error. Still, always sanity-check a corrected DEM/cloud against a stable ground feature before trusting it for py4dgeo (see the plan doc).
- The same spreadsheet also contains SET_1-vs-SET_3 and SET_2-vs-SET_3 comparisons (kept below for reference only). Both have much larger, inconsistent residuals (roughly 0.4–0.7 m stdev on every axis) — i.e. **not** a clean rigid shift, consistent with SET_3 being hand-written field coordinates. No correction path is offered for SET_3.
- Pick **one** set as the project-wide "master" frame and always correct onto it. Recommended: whichever set most of your already-completed months used, to minimize how many files need correcting.
- This assumes each month's own GCP-fit residual is small relative to the ~1.3 m / 1.8 m set1↔set2 discrepancy (true so far: Dec 2024 and Jan 2025 both converged to well under 10 cm against their winning set). If a future month converges loose (tens of cm+) even against its best set, that's a separate problem this constant-shift correction won't fix — resolve that first.
- Correct the **same** point cloud export you actually feed py4dgeo (e.g. after whatever classification/filtering step you use), not an intermediate one that gets re-filtered afterward.

## Setup

Requires `rasterio`, `numpy`, and `laspy` with a LAZ backend (your files are `.laz`, so install `laspy[lazrs]`).

In [1]:
# Run once. Uncomment if these aren't already installed in your environment.
%pip install rasterio numpy "laspy[lazrs]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 812.0 kB/s  0:00:26m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.3/567.3 kB 703.4 kB/s  0:00:00m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [rasterio]7/8 [rasterio]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import math
import numpy as np

## GCP point pairs

Source of truth: the user's own `gcp_comparison_results.xlsx` (Sept 2026), which lists point-by-point horizontal offset, dx (east), dy (north) and dz (up) directly. The first three rows are "known pair" (manually confirmed correspondences); the rest are "auto match" (nearest-point matching, per the spreadsheet's own labelling).

In [ ]:
POINT_PAIRS_SET1_VS_SET2 = [
    dict(label="CAMPAMENTO", row_type="known pair", dx=0.81, dy=0.98, dz=1.77),
    dict(label="1B",         row_type="known pair", dx=0.79, dy=1.10, dz=1.83),
    dict(label="2B",         row_type="known pair", dx=0.75, dy=1.06, dz=1.82),
    dict(label="C",          row_type="auto match", dx=0.81, dy=1.01, dz=1.81),
    dict(label="A1R",        row_type="auto match", dx=0.81, dy=0.98, dz=1.81),
    dict(label="B1R",        row_type="auto match", dx=0.83, dy=0.98, dz=1.83),
    dict(label="B2R",        row_type="auto match", dx=0.84, dy=0.98, dz=1.79),
    dict(label="B3R",        row_type="auto match", dx=0.84, dy=0.98, dz=1.79),
    dict(label="B4R",        row_type="auto match", dx=0.83, dy=1.01, dz=1.87),
    dict(label="C1R",        row_type="auto match", dx=0.83, dy=1.01, dz=1.82),
]

# Verified mean/stdev from the spreadsheet's own AVG/STDV rows for the
# SET_1 vs SET_2 comparison (dx east, dy north, dz up, metres). This is
# what compute_offset() reports and the shift functions ultimately use.
VERIFIED_OFFSET_SET1_VS_SET2 = dict(
    mean=dict(dx=0.814, dy=1.009, dz=1.814),
    std=dict(dx=0.02757, dy=0.04095, dz=0.02757),
)

# For reference only -- NOT used for any correction (see caveats above:
# these are hand-written field coordinates, not a rigid-shift candidate).
POINT_PAIRS_SET1_VS_SET3 = [
    dict(label="A1R", dx=-0.24, dy=-0.53, dz=-0.09),
    dict(label="B1R", dx=0.06, dy=1.71, dz=0.80),
    dict(label="B2R", dx=0.51, dy=0.25, dz=0.53),
    dict(label="B3R", dx=1.43, dy=0.80, dz=0.41),
    dict(label="B4R", dx=1.43, dy=0.45, dz=-0.07),
    dict(label="C1R", dx=0.91, dy=0.88, dz=0.07),
]
POINT_PAIRS_SET2_VS_SET3 = [
    dict(label="A1_HITOCONTROL_QP", dx=-1.05, dy=-1.51, dz=-1.90),
    dict(label="B1_HITOCONTROL_QP", dx=-0.76, dy=0.73, dz=-1.02),
    dict(label="B2_HITOCONTROL_QP", dx=-0.33, dy=-0.73, dz=-1.26),
    dict(label="B3_HITOCONTROL_QP2", dx=0.59, dy=-0.19, dz=-1.38),
    dict(label="C1_HITOCONTROL_QP4", dx=0.07, dy=-0.12, dz=-1.75),
    dict(label="B4_HITOCONTROL_QP7", dx=0.60, dy=-0.56, dz=-1.94),
]

## `compute_offset()` — the verified SET_1 ↔ SET_2 shift

In [ ]:
def compute_offset(verbose=True):
    """Report the verified signed (dx east, dy north, dz up) offset,
    set1 -> set2, plus per-point residuals from the mean. Point values and
    the AVG/STDV cross-check are both taken from the user's verified
    gcp_comparison_results.xlsx. Returns the mean {dx, dy, dz} dict."""
    rows = POINT_PAIRS_SET1_VS_SET2

    dxs = np.array([r["dx"] for r in rows])
    dys = np.array([r["dy"] for r in rows])
    dzs = np.array([r["dz"] for r in rows])

    mean_dx, std_dx = float(dxs.mean()), float(dxs.std())
    mean_dy, std_dy = float(dys.mean()), float(dys.std())
    mean_dz, std_dz = float(dzs.mean()), float(dzs.std())
    horiz_mag = math.hypot(mean_dx, mean_dy)

    if verbose:
        label_h, type_h, dx_h, dy_h, dz_h = "label", "type", "dx east (m)", "dy north (m)", "dz up (m)"
        print(f"{label_h:<12} {type_h:<12} {dx_h:>12} {dy_h:>13} {dz_h:>10}")
        for r in rows:
            lbl, rt = r["label"], r["row_type"]
            print(f"{lbl:<12} {rt:<12} {r['dx']:>12.4f} {r['dy']:>13.4f} {r['dz']:>10.4f}")
        print()
        print(f"Mean offset (set1 -> set2):  dx = {mean_dx:+.4f} m   dy = {mean_dy:+.4f} m   dz = {mean_dz:+.4f} m")
        print(f"Residual stdev:              dx = {std_dx:.4f} m   dy = {std_dy:.4f} m   dz = {std_dz:.4f} m")
        print(f"Horizontal magnitude: {horiz_mag:.3f} m")
        print()
        v = VERIFIED_OFFSET_SET1_VS_SET2
        vm, vs = v["mean"], v["std"]
        print("Cross-check against the spreadsheet's own AVG/STDV row for SET_1 vs SET_2:")
        print(f"  dx={vm['dx']:+.3f}  dy={vm['dy']:+.3f}  dz={vm['dz']:+.3f}   "
              f"stdev: dx={vs['dx']:.4f} dy={vs['dy']:.4f} dz={vs['dz']:.4f}")
        print()
        if max(std_dx, std_dy, std_dz) < 0.05:
            print("Residual scatter is small (<5cm on every axis) -- looks like a genuine")
            print("uniform rigid shift. A constant-offset correction should be valid.")
        else:
            print("WARNING: residual scatter exceeds 5cm on at least one axis -- inspect the")
            print("per-point table above before trusting a constant-shift correction blindly.")

    # Use the spreadsheet's own verified AVG as the value the shift
    # functions apply, since that's the number the user confirmed independently.
    v = VERIFIED_OFFSET_SET1_VS_SET2
    return dict(dx=v["mean"]["dx"], dy=v["mean"]["dy"], dz=v["mean"]["dz"])


def resolve_shift(direction=None, dx=None, dy=None, dz=None):
    """Resolve the (dx, dy, dz) to apply, either from the verified offset
    in a given `direction` ("set1_to_set2" / "set2_to_set1"), or from
    manual dx/dy/dz overrides."""
    if direction:
        offset = compute_offset(verbose=False)
        dx, dy, dz = offset["dx"], offset["dy"], offset["dz"]
        if direction == "set2_to_set1":
            dx, dy, dz = -dx, -dy, -dz
    else:
        if dx is None or dy is None or dz is None:
            raise ValueError("Provide either `direction`, or all of dx/dy/dz manually.")
    return dx, dy, dz

In [ ]:
# Run this to see the offset table and residuals
_ = compute_offset()

## `shift_dem()` — correct a DEM GeoTIFF

Horizontal shift: translates the raster's affine transform only (exact, no resampling/interpolation, no data loss).
Vertical shift: adds `dz` to every valid (non-nodata) pixel.

In [ ]:
def shift_dem(input_path, output_path, dx, dy, dz, nodata_override=None):
    """Apply a rigid (dx, dy, dz) correction to a DEM GeoTIFF.

    dx, dy: horizontal shift in the raster's CRS units (metres, for a UTM
            DEM like this project's WGS 84 / UTM zone 15S exports).
    dz:     vertical shift in metres, added to every valid (non-nodata)
            pixel's elevation value.
    """
    import rasterio

    with rasterio.open(input_path) as src:
        profile = src.profile.copy()
        transform = src.transform
        nodata = nodata_override if nodata_override is not None else src.nodata

        print(f"Input: {input_path}")
        print(f"  CRS: {src.crs}")
        print(f"  Size: {src.width} x {src.height}")
        print(f"  Original nodata: {src.nodata}")
        print(f"  Applying shift: dx={dx:+.4f} m, dy={dy:+.4f} m, dz={dz:+.4f} m")

        new_transform = rasterio.Affine(
            transform.a, transform.b, transform.c + dx,
            transform.d, transform.e, transform.f + dy,
        )
        profile.update(transform=new_transform)

        band_count = src.count
        with rasterio.open(output_path, "w", **profile) as dst:
            for band_idx in range(1, band_count + 1):
                data = src.read(band_idx)
                if nodata is not None:
                    mask = data != nodata
                    data = data.astype("float64", copy=True)
                    data[mask] = data[mask] + dz
                    data = data.astype(profile["dtype"])
                else:
                    data = data + dz
                dst.write(data, band_idx)

    print(f"Written: {output_path}")

### Apply it

Edit the paths and direction below, then run the cell.

In [ ]:
# --- EDIT THESE ---
dem_input_path = "Dec2024_DEM.tif"
dem_output_path = "Dec2024_DEM_on_set2_frame.tif"
dem_direction = "set1_to_set2"   # or "set2_to_set1"
dem_nodata_override = None       # e.g. -9999, only if rasterio reads the wrong nodata value
# ------------------

# dx, dy, dz = resolve_shift(direction=dem_direction)
# shift_dem(dem_input_path, dem_output_path, dx, dy, dz, nodata_override=dem_nodata_override)


## `shift_point_cloud()` — correct a .laz/.las point cloud

No affine transform on a point cloud, so the shift is just added directly to each point's X/Y/Z (`laspy` already gives real-world coordinates, not raw scaled integers). Classification, intensity, RGB, return number, etc. pass through unchanged.

Your files are `.laz` — this needs a LAZ backend (`laspy[lazrs]`, installed above).

In [ ]:
def shift_point_cloud(input_path, output_path, dx, dy, dz):
    """Apply a rigid (dx, dy, dz) correction to a point cloud (.las/.laz).

    dx, dy: horizontal shift, metres (UTM zone 15S for this project).
    dz:     vertical shift, metres, added to every point's Z.
    """
    import laspy

    with laspy.open(input_path) as src_reader:
        las = src_reader.read()

        print(f"Input: {input_path}")
        print(f"  Point format: {las.header.point_format}")
        print(f"  Point count: {las.header.point_count}")
        print(f"  CRS: {las.header.parse_crs()}")
        print(f"  Applying shift: dx={dx:+.4f} m, dy={dy:+.4f} m, dz={dz:+.4f} m")

        las.x = las.x + dx
        las.y = las.y + dy
        las.z = las.z + dz

        las.write(output_path)

    print(f"Written: {output_path}")

### Apply it

Edit the paths and direction below, then run the cell.

In [ ]:
# --- EDIT THESE ---
cloud_input_path = "Dec2024_cloud.laz"
cloud_output_path = "Dec2024_cloud_on_set2_frame.laz"
cloud_direction = "set1_to_set2"   # or "set2_to_set1"
# ------------------

# dx, dy, dz = resolve_shift(direction=cloud_direction)
# shift_point_cloud(cloud_input_path, cloud_output_path, dx, dy, dz)


## Sanity check

Before trusting a corrected DEM/cloud for py4dgeo, verify it against a stable, unchanging ground feature (a building corner, a rock outcrop, a fixed structure) between two months now on the same frame — confirm they actually line up, not just that the numbers ran without error.